# W1 Day5 (04/11 周五): [实战] micrograd 引擎

## 学习目标
- **精读**: Karpathy micrograd: Value 类 / 计算图 / 反向传播 / 优化器
- **实践**: 从零手写自动微分引擎 + 训练验证
- **产出物**: 完整的 micrograd 实现 (可发布 GitHub)
- **时长**: 4h (实战日)

---

## 目录
1. [为什么手写 autograd？](#1)
2. [Value 类: 标量级自动微分](#2)
3. [支持更多运算](#3)
4. [计算图可视化](#4)
5. [反向传播: 拓扑排序 + 链式法则](#5)
6. [与 PyTorch 对比验证](#6)
7. [构建神经网络层: Neuron / Layer / MLP](#7)
8. [训练: 二分类任务](#8)
9. [扩展: 更多激活函数 + 损失函数](#9)
10. [完整 micrograd 库 + 总结](#10)

---
## 1. 为什么手写 autograd？ <a id='1'></a>

### 目标
理解 PyTorch autograd 的**本质**: 从零构建一个能自动计算梯度的系统。

### micrograd 的核心思想

1. 每个 `Value` 对象包装一个标量值
2. 每次运算 (+, *, ...) 创建新的 `Value`，同时记录**如何计算梯度**
3. 调用 `.backward()` 时，从输出向输入**反向传播**梯度

```
前向: a, b → c = a * b → d = c + a → L = f(d)

反向: dL/dd → dL/dc, dL/da₂ → dL/da₁, dL/db → 累加 dL/da = dL/da₁ + dL/da₂
```

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

---
## 2. Value 类: 标量级自动微分 <a id='2'></a>

从最简单的加法和乘法开始。

In [ ]:
class Value:
    """
    micrograd 核心: 标量值 + 自动微分
    
    每个 Value 存储:
    - data: 标量值
    - grad: 梯度 (dL/d_self)
    - _backward: 反向传播函数 (闭包)
    - _prev: 创建此 Value 的子节点
    - _op: 创建此 Value 的运算符 (用于可视化)
    """
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0  # 初始梯度为 0
        self._backward = lambda: None  # 默认无操作
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    # ========== 运算符重载 ==========
    
    def __add__(self, other):
        """加法: z = self + other"""
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            # dz/d(self) = 1, dz/d(other) = 1
            # 链式法则: dL/d(self) += dL/dz * dz/d(self) = out.grad * 1
            self.grad += out.grad * 1.0
            other.grad += out.grad * 1.0
        out._backward = _backward
        
        return out
    
    def __mul__(self, other):
        """乘法: z = self * other"""
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            # dz/d(self) = other.data, dz/d(other) = self.data
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        
        return out
    
    def __pow__(self, other):
        """幂运算: z = self ** other (other 为常数)"""
        assert isinstance(other, (int, float)), "只支持常数指数"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            # dz/d(self) = other * self.data^(other-1)
            self.grad += out.grad * (other * self.data ** (other - 1))
        out._backward = _backward
        
        return out
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __truediv__(self, other):
        return self * (other ** -1) if isinstance(other, Value) else self * (Value(other) ** -1)
    
    def __radd__(self, other):  # other + self
        return self + other
    
    def __rmul__(self, other):  # other * self
        return self * other
    
    def __rsub__(self, other):  # other - self
        return Value(other) + (-self)
    
    def __rtruediv__(self, other):  # other / self
        return Value(other) * (self ** -1)
    
    # ========== 激活函数 ==========
    
    def relu(self):
        """ReLU 激活"""
        out = Value(max(0, self.data), (self,), 'ReLU')
        
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        
        return out
    
    def tanh(self):
        """Tanh 激活"""
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            # d(tanh)/dx = 1 - tanh²(x)
            self.grad += out.grad * (1 - t ** 2)
        out._backward = _backward
        
        return out
    
    def exp(self):
        """指数函数"""
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        
        def _backward():
            self.grad += out.grad * e
        out._backward = _backward
        
        return out
    
    def log(self):
        """对数函数"""
        out = Value(math.log(self.data), (self,), 'log')
        
        def _backward():
            self.grad += out.grad * (1.0 / self.data)
        out._backward = _backward
        
        return out
    
    def sigmoid(self):
        """Sigmoid 激活"""
        s = 1.0 / (1.0 + math.exp(-self.data))
        out = Value(s, (self,), 'σ')
        
        def _backward():
            self.grad += out.grad * s * (1 - s)
        out._backward = _backward
        
        return out
    
    # ========== 反向传播 ==========
    
    def backward(self):
        """
        反向传播: 拓扑排序 + 逆序执行 _backward
        """
        # Step 1: 拓扑排序 (DFS)
        topo = []
        visited = set()
        
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        
        build_topo(self)
        
        # Step 2: 从输出到输入反向传播
        self.grad = 1.0  # dL/dL = 1
        for v in reversed(topo):
            v._backward()
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"


print("=" * 60)
print("Value 类基础测试")
print("=" * 60)

# 测试: z = (a + b) * c
a = Value(2.0, label='a')
b = Value(3.0, label='b')
c = Value(-4.0, label='c')

d = a + b       # d = 5
e = d * c       # e = -20
e.backward()

print(f"  a = {a}")
print(f"  b = {b}")
print(f"  c = {c}")
print(f"  e = (a+b)*c = {e}")
print(f"\n  de/da = c = {a.grad:.1f} (理论: {c.data})")
print(f"  de/db = c = {b.grad:.1f} (理论: {c.data})")
print(f"  de/dc = a+b = {c.grad:.1f} (理论: {a.data + b.data})")

---
## 3. 支持更多运算 <a id='3'></a>

验证所有运算的梯度正确性。

In [ ]:
print("=" * 60)
print("各运算梯度验证")
print("=" * 60)

def numerical_grad(f, x, eps=1e-7):
    """数值梯度 (中心差分)"""
    return (f(x + eps) - f(x - eps)) / (2 * eps)

def test_op(name, f_value, f_float, x_val, y_val=None):
    """测试一个运算的梯度"""
    x = Value(x_val)
    if y_val is not None:
        y = Value(y_val)
        z = f_value(x, y)
        z.backward()
        num_dx = numerical_grad(lambda v: f_float(v, y_val), x_val)
        num_dy = numerical_grad(lambda v: f_float(x_val, v), y_val)
        ok = abs(x.grad - num_dx) < 1e-5 and abs(y.grad - num_dy) < 1e-5
        symbol = "✅" if ok else "❌"
        print(f"  {symbol} {name:15s}: dx={x.grad:+.6f} (num={num_dx:+.6f}), "
              f"dy={y.grad:+.6f} (num={num_dy:+.6f})")
    else:
        z = f_value(x)
        z.backward()
        num_dx = numerical_grad(lambda v: f_float(v), x_val)
        ok = abs(x.grad - num_dx) < 1e-5
        symbol = "✅" if ok else "❌"
        print(f"  {symbol} {name:15s}: dx={x.grad:+.6f} (num={num_dx:+.6f})")

# 二元运算
test_op("a + b", lambda a,b: a+b, lambda a,b: a+b, 2.0, 3.0)
test_op("a * b", lambda a,b: a*b, lambda a,b: a*b, 2.0, 3.0)
test_op("a - b", lambda a,b: a-b, lambda a,b: a-b, 2.0, 3.0)
test_op("a / b", lambda a,b: a/b, lambda a,b: a/b, 6.0, 3.0)

# 一元运算
test_op("x**2", lambda x: x**2, lambda x: x**2, 3.0)
test_op("x**3", lambda x: x**3, lambda x: x**3, 2.0)
test_op("relu(x>0)", lambda x: x.relu(), lambda x: max(0,x), 2.0)
test_op("relu(x<0)", lambda x: x.relu(), lambda x: max(0,x), -2.0)
test_op("tanh(x)", lambda x: x.tanh(), lambda x: math.tanh(x), 1.5)
test_op("exp(x)", lambda x: x.exp(), lambda x: math.exp(x), 1.0)
test_op("log(x)", lambda x: x.log(), lambda x: math.log(x), 2.0)
test_op("sigmoid(x)", lambda x: x.sigmoid(), lambda x: 1/(1+math.exp(-x)), 1.0)

# 复合运算
print("\n--- 复合运算 ---")
test_op("x²+2x+1", 
        lambda x: x**2 + 2*x + 1, 
        lambda x: x**2 + 2*x + 1, 3.0)
test_op("tanh(2x+1)",
        lambda x: (2*x + 1).tanh(),
        lambda x: math.tanh(2*x + 1), 0.5)

In [ ]:
# 关键测试: 同一变量多次使用 (梯度累加)
print("\n" + "=" * 60)
print("梯度累加测试 (同一变量多次使用)")
print("=" * 60)

x = Value(3.0)
y = x + x  # y = 2x, dy/dx = 2
y.backward()
print(f"  y = x + x: dy/dx = {x.grad:.1f} (理论: 2.0) {'✅' if abs(x.grad - 2.0) < 1e-6 else '❌'}")

x = Value(3.0)
y = x * x  # y = x², dy/dx = 2x = 6
y.backward()
print(f"  y = x * x: dy/dx = {x.grad:.1f} (理论: 6.0) {'✅' if abs(x.grad - 6.0) < 1e-6 else '❌'}")

x = Value(2.0)
y = x * x + x * 3  # y = x² + 3x, dy/dx = 2x + 3 = 7
y.backward()
print(f"  y = x²+3x: dy/dx = {x.grad:.1f} (理论: 7.0) {'✅' if abs(x.grad - 7.0) < 1e-6 else '❌'}")

print("\n💡 += 操作是关键: 一个节点被多条路径使用时，梯度要累加！")

---
## 4. 计算图可视化 <a id='4'></a>

In [ ]:
def draw_graph(root, figsize=(14, 8)):
    """
    用 matplotlib 绘制计算图 (不依赖 graphviz)
    """
    # 收集所有节点和边
    nodes = []
    edges = []
    visited = set()
    
    def traverse(v, depth=0):
        if id(v) not in visited:
            visited.add(id(v))
            nodes.append((v, depth))
            for child in v._prev:
                edges.append((child, v))
                traverse(child, depth + 1)
    
    traverse(root)
    
    # 按深度分层
    max_depth = max(d for _, d in nodes) if nodes else 0
    layers = {}
    for v, d in nodes:
        depth = max_depth - d
        if depth not in layers:
            layers[depth] = []
        layers[depth].append(v)
    
    # 分配位置
    positions = {}
    for depth, layer_nodes in layers.items():
        for i, v in enumerate(layer_nodes):
            x = depth * 3
            y = -(i - (len(layer_nodes) - 1) / 2) * 2
            positions[id(v)] = (x, y)
    
    # 绘制
    fig, ax = plt.subplots(figsize=figsize)
    
    # 画边
    for child, parent in edges:
        x1, y1 = positions[id(child)]
        x2, y2 = positions[id(parent)]
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    
    # 画节点
    for v, _ in nodes:
        x, y = positions[id(v)]
        label = v.label or v._op or ''
        text = f"{label}\ndata={v.data:.3f}\ngrad={v.grad:.3f}"
        
        color = '#AED6F1' if v._prev else '#A9DFBF'  # 蓝=中间, 绿=叶子
        if v is root:
            color = '#F9E79F'  # 黄=输出
        
        bbox = dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.8)
        ax.text(x, y, text, fontsize=8, ha='center', va='center', bbox=bbox)
    
    ax.set_xlim(-1, max_depth * 3 + 1)
    ax.axis('off')
    ax.set_title('计算图 (绿=叶子输入, 蓝=中间运算, 黄=输出)', fontsize=12)
    plt.tight_layout()
    plt.show()


# 构建并可视化一个表达式
print("=" * 60)
print("计算图可视化")
print("=" * 60)

x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
b = Value(6.881, label='b')

# 一个 neuron 的前向: tanh(x1*w1 + x2*w2 + b)
x1w1 = x1 * w1
x2w2 = x2 * w2
s = x1w1 + x2w2 + b
o = s.tanh()
o.label = 'output'

o.backward()

print(f"  表达式: o = tanh(x1*w1 + x2*w2 + b)")
print(f"  o = {o.data:.4f}")
print(f"  do/dx1 = {x1.grad:.4f}")
print(f"  do/dw1 = {w1.grad:.4f}")
print(f"  do/dx2 = {x2.grad:.4f}")
print(f"  do/dw2 = {w2.grad:.4f}")

draw_graph(o, figsize=(16, 6))

---
## 5. 反向传播: 拓扑排序 + 链式法则 <a id='5'></a>

### 为什么需要拓扑排序？

反向传播要求: **先算后面的节点，再算前面的节点**。

拓扑排序保证: 每个节点在其所有消费者之后被处理。

```
前向: x → a → b → L
拓扑: [x, a, b, L]
反向: L → b → a → x (逆序)
```

In [ ]:
# 拓扑排序可视化
print("=" * 60)
print("拓扑排序 + 反向传播逐步执行")
print("=" * 60)

# 简单表达式: L = (a * b + c)^2
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

d = a * b; d.label = 'd=a*b'
e = d + c; e.label = 'e=d+c'
L = e ** 2; L.label = 'L=e²'

# 手动逐步反向传播
print(f"\n前向:")
print(f"  d = a*b = {d.data}")
print(f"  e = d+c = {e.data}")
print(f"  L = e²  = {L.data}")

print(f"\n手动反向传播:")
# dL/dL = 1
L.grad = 1.0
print(f"  Step 0: dL/dL = {L.grad}")

# dL/de = 2e = 2*4 = 8
e.grad = L.grad * 2 * e.data
print(f"  Step 1: dL/de = {L.grad} * 2*{e.data} = {e.grad}")

# dL/dd = dL/de * de/dd = 8 * 1 = 8
d.grad = e.grad * 1.0
# dL/dc = dL/de * de/dc = 8 * 1 = 8
c.grad = e.grad * 1.0
print(f"  Step 2: dL/dd = {d.grad}, dL/dc = {c.grad}")

# dL/da = dL/dd * dd/da = 8 * b = 8 * -3 = -24
a.grad = d.grad * b.data
# dL/db = dL/dd * dd/db = 8 * a = 8 * 2 = 16
b.grad = d.grad * a.data
print(f"  Step 3: dL/da = {a.grad}, dL/db = {b.grad}")

# 用 backward 验证
print(f"\nautomatic backward 验证:")
a2 = Value(2.0)
b2 = Value(-3.0)
c2 = Value(10.0)
L2 = ((a2 * b2) + c2) ** 2
L2.backward()
print(f"  dL/da = {a2.grad} {'✅' if abs(a2.grad - a.grad) < 1e-6 else '❌'}")
print(f"  dL/db = {b2.grad} {'✅' if abs(b2.grad - b.grad) < 1e-6 else '❌'}")
print(f"  dL/dc = {c2.grad} {'✅' if abs(c2.grad - c.grad) < 1e-6 else '❌'}")

---
## 6. 与 PyTorch 对比验证 <a id='6'></a>

In [ ]:
import torch

print("=" * 60)
print("micrograd vs PyTorch 对比验证")
print("=" * 60)

def compare_with_pytorch(expr_name, build_mg, build_pt):
    """对比 micrograd 和 PyTorch 的梯度"""
    # micrograd
    mg_vals, mg_result = build_mg()
    mg_result.backward()
    mg_grads = {name: v.grad for name, v in mg_vals.items()}
    
    # PyTorch
    pt_vals, pt_result = build_pt()
    pt_result.backward()
    pt_grads = {name: v.grad.item() for name, v in pt_vals.items()}
    
    # 对比
    all_ok = True
    print(f"\n  {expr_name}: result mg={mg_result.data:.6f}, pt={pt_result.item():.6f}")
    for name in mg_grads:
        mg_g = mg_grads[name]
        pt_g = pt_grads[name]
        ok = abs(mg_g - pt_g) < 1e-5
        all_ok = all_ok and ok
        symbol = "✅" if ok else "❌"
        print(f"    {symbol} d/d{name}: mg={mg_g:.6f}, pt={pt_g:.6f}")
    return all_ok

# Test 1: 多项式
compare_with_pytorch(
    "z = 3x² + 2xy - y³",
    lambda: (
        vals := {'x': Value(2.0), 'y': Value(3.0)},
        3 * vals['x']**2 + 2 * vals['x'] * vals['y'] - vals['y']**3
    )[-1:][0] if False else (
        (d := {'x': Value(2.0), 'y': Value(3.0)}),
        3 * d['x']**2 + 2 * d['x'] * d['y'] - d['y']**3
    ),
    lambda: (
        (d := {'x': torch.tensor(2.0, requires_grad=True), 
               'y': torch.tensor(3.0, requires_grad=True)}),
        3 * d['x']**2 + 2 * d['x'] * d['y'] - d['y']**3
    ),
)

# Test 2: 神经元
compare_with_pytorch(
    "neuron = tanh(w1*x1 + w2*x2 + b)",
    lambda: (
        (d := {'x1': Value(2.0), 'x2': Value(0.5), 
               'w1': Value(-3.0), 'w2': Value(1.0), 'b': Value(1.5)}),
        (d['w1']*d['x1'] + d['w2']*d['x2'] + d['b']).tanh()
    ),
    lambda: (
        (d := {'x1': torch.tensor(2.0, requires_grad=True),
               'x2': torch.tensor(0.5, requires_grad=True),
               'w1': torch.tensor(-3.0, requires_grad=True),
               'w2': torch.tensor(1.0, requires_grad=True),
               'b': torch.tensor(1.5, requires_grad=True)}),
        torch.tanh(d['w1']*d['x1'] + d['w2']*d['x2'] + d['b'])
    ),
)

---
## 7. 构建神经网络层: Neuron / Layer / MLP <a id='7'></a>

用 Value 类构建完整的神经网络。

In [ ]:
class Module:
    """所有模块的基类 (类似 nn.Module)"""
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    
    def parameters(self):
        return []


class Neuron(Module):
    """
    单个神经元: y = activation(w·x + b)
    """
    def __init__(self, n_in, activation='relu'):
        # 随机初始化权重 (Xavier-like)
        scale = (2.0 / n_in) ** 0.5
        self.w = [Value(random.uniform(-1, 1) * scale) for _ in range(n_in)]
        self.b = Value(0.0)
        self.activation = activation
    
    def __call__(self, x):
        # w · x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        
        if self.activation == 'relu':
            return act.relu()
        elif self.activation == 'tanh':
            return act.tanh()
        elif self.activation == 'sigmoid':
            return act.sigmoid()
        elif self.activation == 'none' or self.activation is None:
            return act
        else:
            raise ValueError(f"Unknown activation: {self.activation}")
    
    def parameters(self):
        return self.w + [self.b]
    
    def __repr__(self):
        return f"Neuron({len(self.w)}, {self.activation})"


class Layer(Module):
    """一层神经元"""
    def __init__(self, n_in, n_out, activation='relu'):
        self.neurons = [Neuron(n_in, activation) for _ in range(n_out)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
    
    def __repr__(self):
        return f"Layer({', '.join(str(n) for n in self.neurons)})"


class MLP(Module):
    """
    多层感知机
    
    Example:
        mlp = MLP(3, [4, 4, 1])  # 3→4→4→1
    """
    def __init__(self, n_in, n_outs, activation='relu'):
        sizes = [n_in] + n_outs
        self.layers = []
        for i in range(len(n_outs)):
            # 最后一层不用激活
            act = activation if i < len(n_outs) - 1 else 'none'
            self.layers.append(Layer(sizes[i], sizes[i+1], act))
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
    
    def __repr__(self):
        layers_str = '\n  '.join(str(l) for l in self.layers)
        return f"MLP(\n  {layers_str}\n)"


# 测试
print("=" * 60)
print("MLP 结构")
print("=" * 60)

mlp = MLP(3, [4, 4, 1])
print(f"\n{mlp}")
print(f"参数量: {len(mlp.parameters())}")

# 前向测试
x = [Value(1.0), Value(2.0), Value(-1.0)]
y = mlp(x)
print(f"\n输入: [1.0, 2.0, -1.0]")
print(f"输出: {y}")

# 反向传播
y.backward()
print(f"\n前几个参数的梯度:")
for i, p in enumerate(mlp.parameters()[:5]):
    print(f"  param[{i}]: data={p.data:.4f}, grad={p.grad:.4f}")

---
## 8. 训练: 二分类任务 <a id='8'></a>

在一个经典的月牙形数据集上训练 MLP。

In [ ]:
# 生成月牙形数据
from sklearn.datasets import make_moons

np.random.seed(42)
X_np, y_np = make_moons(n_samples=100, noise=0.1)
y_np = y_np * 2 - 1  # 转为 {-1, +1}

# 可视化数据
plt.figure(figsize=(8, 6))
plt.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='blue', label='+1', s=30)
plt.scatter(X_np[y_np==-1, 0], X_np[y_np==-1, 1], c='red', label='-1', s=30)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Moon 数据集 (二分类)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 训练 MLP
print("=" * 60)
print("训练 MLP (micrograd)")
print("=" * 60)

random.seed(42)
model = MLP(2, [16, 16, 1], activation='relu')
print(f"模型参数量: {len(model.parameters())}")

# 超参数
lr = 0.05
n_epochs = 50

losses = []
accuracies = []

for epoch in range(n_epochs):
    # 前向传播: 对每个样本计算输出
    preds = [model([Value(x[0]), Value(x[1])]) for x in X_np]
    
    # Hinge Loss (SVM loss): max(0, 1 - y*pred)
    data_loss = sum(
        (1 + (-yi) * pred_i).relu() 
        for yi, pred_i in zip(y_np, preds)
    ) * (1.0 / len(y_np))
    
    # L2 正则化
    reg_loss = sum(p * p for p in model.parameters()) * 1e-4
    total_loss = data_loss + reg_loss
    
    # 准确率
    acc = sum(
        (pred_i.data > 0) == (yi > 0)
        for pred_i, yi in zip(preds, y_np)
    ) / len(y_np)
    
    losses.append(total_loss.data)
    accuracies.append(acc)
    
    # 反向传播
    model.zero_grad()
    total_loss.backward()
    
    # SGD 更新
    for p in model.parameters():
        p.data -= lr * p.grad
    
    # 学习率衰减
    if (epoch + 1) % 20 == 0:
        lr *= 0.5
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {total_loss.data:.4f} | "
              f"Acc: {acc*100:.1f}% | LR: {lr:.4f}")

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('训练 Loss')

# Accuracy
axes[1].plot([a*100 for a in accuracies], 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('训练准确率')
axes[1].set_ylim(0, 105)

# 决策边界
ax = axes[2]
h = 0.1
x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# 预测网格点
grid_points = np.c_[xx.ravel(), yy.ravel()]
Z = np.array([model([Value(p[0]), Value(p[1])]).data for p in grid_points])
Z = Z.reshape(xx.shape)

ax.contourf(xx, yy, Z, levels=[-1, 0, 1], colors=['#FFCCCC', '#CCCCFF'], alpha=0.5)
ax.contour(xx, yy, Z, levels=[0], colors='black', linewidths=2)
ax.scatter(X_np[y_np==1, 0], X_np[y_np==1, 1], c='blue', s=30, edgecolors='k', linewidths=0.5)
ax.scatter(X_np[y_np==-1, 0], X_np[y_np==-1, 1], c='red', s=30, edgecolors='k', linewidths=0.5)
ax.set_title(f'决策边界 (Acc={accuracies[-1]*100:.1f}%)')
ax.set_xlabel('x1')
ax.set_ylabel('x2')

plt.suptitle('micrograd 训练结果', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. 扩展: 更多损失函数 <a id='9'></a>

除了 Hinge Loss，实现 Binary Cross-Entropy 和 MSE。

In [ ]:
# 损失函数
def binary_cross_entropy(pred, target):
    """
    BCE Loss: -[y*log(σ(pred)) + (1-y)*log(1-σ(pred))]
    target: 0 或 1
    """
    prob = pred.sigmoid()
    # 数值稳定: 避免 log(0)
    eps = Value(1e-7)
    if target == 1:
        return -(prob + eps).log()
    else:
        return -(1 - prob.data + eps.data)
        # 更稳定的实现
        one_minus = Value(1.0) + (Value(-1.0) * prob)
        return -(one_minus + eps).log()

def mse_loss(pred, target):
    """MSE Loss: (pred - target)²"""
    return (pred - target) ** 2


# 用 BCE 重新训练
print("=" * 60)
print("BCE Loss 训练")
print("=" * 60)

random.seed(42)
model_bce = MLP(2, [16, 16, 1], activation='relu')
y_01 = (y_np + 1) / 2  # 转为 {0, 1}

lr = 0.05
bce_losses = []
bce_accs = []

for epoch in range(50):
    preds = [model_bce([Value(x[0]), Value(x[1])]) for x in X_np]
    probs = [p.sigmoid() for p in preds]
    
    # BCE
    total_loss = Value(0.0)
    for prob, y in zip(probs, y_01):
        if y == 1:
            total_loss = total_loss + (-(prob + Value(1e-7)).log())
        else:
            one_minus = Value(1.0) + Value(-1.0) * prob
            total_loss = total_loss + (-(one_minus + Value(1e-7)).log())
    total_loss = total_loss * (1.0 / len(y_01))
    
    # 正则化
    reg = sum(p * p for p in model_bce.parameters()) * 1e-4
    total_loss = total_loss + reg
    
    acc = sum((p.data > 0.5) == (y == 1) for p, y in zip(probs, y_01)) / len(y_01)
    bce_losses.append(total_loss.data)
    bce_accs.append(acc)
    
    model_bce.zero_grad()
    total_loss.backward()
    
    for p in model_bce.parameters():
        p.data -= lr * p.grad
    
    if (epoch + 1) % 20 == 0:
        lr *= 0.5
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d} | BCE: {total_loss.data:.4f} | Acc: {acc*100:.1f}%")

In [ ]:
# 与 PyTorch 训练对比
print("\n" + "=" * 60)
print("PyTorch 对比训练 (验证 micrograd 效果)")
print("=" * 60)

import torch
import torch.nn as nn

torch.manual_seed(42)
X_t = torch.tensor(X_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

pt_model = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1)
)

pt_opt = torch.optim.SGD(pt_model.parameters(), lr=0.05)
pt_losses = []

for epoch in range(50):
    pt_opt.zero_grad()
    out = pt_model(X_t)
    # Hinge loss
    loss = torch.relu(1 - y_t * out).mean() + 1e-4 * sum(p.pow(2).sum() for p in pt_model.parameters())
    loss.backward()
    pt_opt.step()
    pt_losses.append(loss.item())
    
    if (epoch + 1) % 20 == 0:
        for g in pt_opt.param_groups:
            g['lr'] *= 0.5
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        acc = ((out > 0).float() == (y_t > 0).float()).float().mean()
        print(f"  [PyTorch] Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Acc: {acc*100:.1f}%")

# 对比 loss 曲线
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, 'b-', linewidth=2, label='micrograd (Hinge)')
ax.plot(pt_losses, 'r--', linewidth=2, label='PyTorch (Hinge)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('micrograd vs PyTorch 训练对比')
ax.legend()
plt.tight_layout()
plt.show()

---
## 10. 完整 micrograd 库 + 总结 <a id='10'></a>

In [ ]:
# 完整 micrograd API 汇总
print("=" * 60)
print("micrograd 完整 API")
print("=" * 60)

print("""
Value 类 (标量自动微分):
  运算:     +, -, *, /, **
  激活:     .relu(), .tanh(), .sigmoid(), .exp(), .log()
  反向:     .backward()
  属性:     .data, .grad

Module 基类:
  .parameters()  → 所有参数列表
  .zero_grad()   → 梯度清零

Neuron(n_in, activation):  单个神经元
Layer(n_in, n_out, act):   一层神经元
MLP(n_in, [h1, h2, out]): 多层感知机

训练循环:
  1. preds = [model(x) for x in data]
  2. loss = compute_loss(preds, targets)
  3. model.zero_grad()
  4. loss.backward()
  5. for p in model.parameters(): p.data -= lr * p.grad
""")

# 代码行数统计
print("核心代码行数估计:")
print("  Value 类:     ~120 行")
print("  Module 基类:   ~10 行")
print("  Neuron:        ~25 行")
print("  Layer:         ~15 行")
print("  MLP:           ~20 行")
print("  总计:          ~190 行")
print("\n💡 PyTorch 的 autograd 本质上做的就是这件事，只是:")
print("   - Tensor (多维数组) 替代 Value (标量)")
print("   - C++/CUDA 实现替代 Python")
print("   - 支持更多运算符和优化")

### 今日核心收获

1. **Value 类**: 每个运算记录 `_backward` 闭包，反向时执行
2. **梯度累加**: 同一变量被多条路径使用时，梯度要 `+=`
3. **拓扑排序**: 保证反向传播的正确顺序
4. **链式法则**: `∂L/∂x = ∂L/∂z · ∂z/∂x`，每个运算只需知道自己的局部梯度
5. **从 Value 到 PyTorch**: Value→Tensor, Neuron→Linear, backward→autograd

### 面试关键问题

1. **autograd 的核心数据结构？** → 计算图 DAG, 每个节点存 grad_fn
2. **为什么需要拓扑排序？** → 保证先算下游梯度再算上游
3. **梯度为什么要累加？** → 一个变量可能出现在多条计算路径中
4. **动态图 vs 静态图？** → 动态图每次前向重建; 灵活但略慢

### 明日预习: CNN + ResNet + ViT
- 卷积的感受野
- 残差连接的恒等映射直觉
- ViT 的 Patch Embedding

In [ ]:
print("\n" + "=" * 60)
print("W1 Day5 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ Value 类: +, -, *, /, ** 运算符重载 + 梯度闭包
  ✅ 激活函数: ReLU, Tanh, Sigmoid, Exp, Log
  ✅ 全部运算 数值梯度验证 (12 种运算 + 复合运算)
  ✅ 梯度累加测试 (x+x, x*x, x²+3x)
  ✅ 计算图可视化 (matplotlib)
  ✅ 拓扑排序 + 反向传播逐步推导
  ✅ 与 PyTorch 对比验证 (多项式 + 神经元)
  ✅ Module / Neuron / Layer / MLP 完整实现
  ✅ Moon 数据集训练 (Hinge Loss + L2 正则)
  ✅ 决策边界可视化
  ✅ BCE Loss 训练
  ✅ micrograd vs PyTorch Loss 曲线对比
""")